## Data Engineering Frl

In [1]:
!pip install skimpy mlflow pyngrok xgboost lightgbm econml dowhy causalml shap

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 6.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 7.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 4.6 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of causalml to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 83.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 92.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 77.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 97.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 403.1/403.1 kB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 97.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.9/76.9 kB 7.5 MB/s e

In [2]:
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import skimpy as sk
import mlflow
import mlflow.sklearn
from pyngrok import ngrok


from sklearn.preprocessing import *
from sklearn.impute import SimpleImputer
from sklearn.model_selection import *
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.pipeline import Pipeline

#Model Export
import joblib

In [3]:
for files in os.scandir("/content/drive/MyDrive/march-machine-learning-mania-2026"):
  print(files)

<DirEntry 'Cities.csv'>
<DirEntry 'MGameCities.csv'>
<DirEntry 'MConferenceTourneyGames.csv'>
<DirEntry 'Conferences.csv'>
<DirEntry 'MNCAATourneyDetailedResults.csv'>
<DirEntry 'SampleSubmissionStage2.csv'>
<DirEntry 'WNCAATourneyCompactResults.csv'>
<DirEntry 'MNCAATourneyCompactResults.csv'>
<DirEntry 'MSecondaryTourneyTeams.csv'>
<DirEntry 'MNCAATourneySlots.csv'>
<DirEntry 'MTeamSpellings.csv'>
<DirEntry 'WRegularSeasonCompactResults.csv'>
<DirEntry 'MNCAATourneySeedRoundSlots.csv'>
<DirEntry 'MSecondaryTourneyCompactResults.csv'>
<DirEntry 'MMasseyOrdinals.csv'>
<DirEntry 'MNCAATourneySeeds.csv'>
<DirEntry 'MTeams.csv'>
<DirEntry 'WGameCities.csv'>
<DirEntry 'SampleSubmissionStage1.csv'>
<DirEntry 'MTeamConferences.csv'>
<DirEntry 'MTeamCoaches.csv'>
<DirEntry 'MRegularSeasonCompactResults.csv'>
<DirEntry 'MSeasons.csv'>
<DirEntry 'WNCAATourneySeeds.csv'>
<DirEntry 'WRegularSeasonDetailedResults.csv'>
<DirEntry 'MRegularSeasonDetailedResults.csv'>
<DirEntry 'WNCAATourneyDetailedR

In [18]:
"""
March Machine Learning Mania - Training & Testing Data Generation Script
This script processes all CSV files to create feature-rich datasets for training and testing
"""

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

class MarchMadnessDataGenerator:
    def __init__(self, data_path=''):
        self.data_path = data_path
        self.seasons = range(2003, 2024)  # Adjust based on your data

    def load_all_data(self):
        """Load all CSV files into dataframes"""
        print("Loading all data files...")

        # Teams data
        self.teams_m = pd.read_csv(f'{self.data_path}MTeams.csv')
        self.teams_w = pd.read_csv(f'{self.data_path}WTeams.csv')

        # Regular season results
        self.reg_season_m = pd.read_csv(f'{self.data_path}MRegularSeasonCompactResults.csv')
        self.reg_season_w = pd.read_csv(f'{self.data_path}WRegularSeasonCompactResults.csv')

        # Try to load detailed results if available
        try:
            self.reg_season_m_detailed = pd.read_csv(f'{self.data_path}MRegularSeasonDetailedResults.csv')
            self.reg_season_w_detailed = pd.read_csv(f'{self.data_path}WRegularSeasonDetailedResults.csv')
            self.has_detailed = True
        except:
            self.has_detailed = False
            print("Detailed regular season results not found, using compact results only")

        # Tournament results
        self.tourney_m = pd.read_csv(f'{self.data_path}MNCAATourneyCompactResults.csv')
        self.tourney_w = pd.read_csv(f'{self.data_path}WNCAATourneyCompactResults.csv')

        # Tournament seeds
        self.seeds_m = pd.read_csv(f'{self.data_path}MNCAATourneySeeds.csv')
        self.seeds_w = pd.read_csv(f'{self.data_path}WNCAATourneySeeds.csv')

        # Team conferences
        self.team_conferences_m = pd.read_csv(f'{self.data_path}MTeamConferences.csv')
        self.team_conferences_w = pd.read_csv(f'{self.data_path}WTeamConferences.csv')

        # Conferences
        self.conferences = pd.read_csv(f'{self.data_path}Conferences.csv')

        # Sample submission for test data reference
        try:
            self.sample_submission_stage1 = pd.read_csv(f'{self.data_path}SampleSubmissionStage1.csv')
            print("SampleSubmissionStage1.csv loaded")
        except:
            self.sample_submission_stage1 = None
            print("SampleSubmissionStage1.csv not found")

        try:
            self.sample_submission_stage2 = pd.read_csv(f'{self.data_path}SampleSubmissionStage2.csv')
            print("SampleSubmissionStage2.csv loaded")
        except:
            self.sample_submission_stage2 = None
            print("SampleSubmissionStage2.csv not found")

        # Massey Ordinals (ratings)
        try:
            self.massey = pd.read_csv(f'{self.data_path}MMasseyOrdinals.csv')
            print(f"Massey Ordinals loaded with columns: {self.massey.columns.tolist()}")
        except:
            self.massey = None
            print("Massey Ordinals not found")

        print("All files loaded successfully!")

    def extract_seed_number(self, seed):
        """Extract numeric seed from seed string (e.g., 'W01' -> 1)"""
        if pd.isna(seed):
            return 16  # Default to worst seed if missing
        return int(seed[1:3])

    def extract_seed_region(self, seed):
        """Extract region from seed string"""
        if pd.isna(seed):
            return 'Z'
        return seed[0]

    def create_team_features(self, gender='M'):
        """Create features for each team per season"""
        print(f"Creating team features for {gender}...")

        if gender == 'M':
            reg_season = self.reg_season_m
            teams = self.teams_m
            team_conferences = self.team_conferences_m
            seeds = self.seeds_m
        else:
            reg_season = self.reg_season_w
            teams = self.teams_w
            team_conferences = self.team_conferences_w
            seeds = self.seeds_w

        team_features = []

        for season in self.seasons:
            season_games = reg_season[reg_season['Season'] == season]
            if len(season_games) == 0:
                continue

            # Get all teams that played this season
            teams_in_season = pd.unique(pd.concat([season_games['WTeamID'], season_games['LTeamID']]))

            for team_id in teams_in_season:
                # Team's games this season
                team_wins = season_games[season_games['WTeamID'] == team_id]
                team_losses = season_games[season_games['LTeamID'] == team_id]
                team_games = pd.concat([team_wins, team_losses])

                # Basic stats
                num_games = len(team_games)
                num_wins = len(team_wins)
                win_pct = num_wins / num_games if num_games > 0 else 0

                # Points for/against
                points_for = team_wins['WScore'].sum() + team_losses['LScore'].sum()
                points_against = team_wins['LScore'].sum() + team_losses['WScore'].sum()
                avg_points_for = points_for / num_games if num_games > 0 else 0
                avg_points_against = points_against / num_games if num_games > 0 else 0

                # Conference
                conference_row = team_conferences[(team_conferences['TeamID'] == team_id) &
                                                 (team_conferences['Season'] == season)]
                conference = conference_row['ConfAbbrev'].values[0] if len(conference_row) > 0 else 'UNK'

                # Seed (if available for tournament)
                seed_row = seeds[(seeds['TeamID'] == team_id) & (seeds['Season'] == season)]
                if len(seed_row) > 0:
                    seed_str = seed_row['Seed'].values[0]
                    seed_num = self.extract_seed_number(seed_str)
                    seed_region = self.extract_seed_region(seed_str)
                else:
                    seed_num = np.nan
                    seed_region = np.nan

                team_features.append({
                    'Season': season,
                    'TeamID': team_id,
                    'TeamName': teams[teams['TeamID'] == team_id]['TeamName'].values[0] if len(teams[teams['TeamID'] == team_id]) > 0 else f"Team_{team_id}",
                    'Conference': conference,
                    'NumGames': num_games,
                    'NumWins': num_wins,
                    'WinPct': win_pct,
                    'AvgPointsFor': avg_points_for,
                    'AvgPointsAgainst': avg_points_against,
                    'PointDiff': avg_points_for - avg_points_against,
                    'Seed': seed_num,
                    'SeedRegion': seed_region
                })

        return pd.DataFrame(team_features)

    def add_massey_ratings(self, features_df, gender='M'):
        """Add Massey ratings to team features"""
        if gender != 'M' or self.massey is None:
            return features_df

        print("Adding Massey ratings...")

        # Check actual column names in the Massey file
        massey_columns = self.massey.columns.tolist()

        # Determine the correct column names
        if 'SysName' in massey_columns:
            system_col = 'SysName'
        elif 'SystemName' in massey_columns:
            system_col = 'SystemName'
        else:
            print("Could not find system name column in Massey data")
            return features_df

        if 'OrdinalRank' in massey_columns:
            rank_col = 'OrdinalRank'
        elif 'Rank' in massey_columns:
            rank_col = 'Rank'
        else:
            print("Could not find rank column in Massey data")
            return features_df

        # Use most common rating systems
        rating_systems = ['POM', 'SAG', 'MOR', 'WLK']

        for system in rating_systems:
            system_ratings = self.massey[self.massey[system_col] == system]
            ratings_dict = {}

            for _, row in system_ratings.iterrows():
                key = (row['Season'], row['TeamID'])
                ratings_dict[key] = row[rank_col]

            features_df[f'Massey_{system}'] = features_df.apply(
                lambda x: ratings_dict.get((x['Season'], x['TeamID']), np.nan), axis=1
            )

        return features_df

    def create_game_features(self, gender='M', for_training=True):
        """Create features for tournament games"""
        print(f"Creating game features for {gender} {'training' if for_training else 'testing'}...")

        if gender == 'M':
            tourney = self.tourney_m
            reg_season = self.reg_season_m
        else:
            tourney = self.tourney_w
            reg_season = self.reg_season_w

        # Create team features first
        team_features = self.create_team_features(gender)
        team_features = self.add_massey_ratings(team_features, gender)

        games = []

        if for_training:
            # Use tournament results for training
            game_iter = tourney.iterrows()
            data_source = "tournament"
        else:
            # Use sample submission for testing
            if self.sample_submission_stage2 is not None:
                sample_sub = self.sample_submission_stage2
            elif self.sample_submission_stage1 is not None:
                sample_sub = self.sample_submission_stage1
            else:
                print("No sample submission found for test data")
                return pd.DataFrame()

            game_iter = sample_sub.iterrows()
            data_source = "sample submission"

        print(f"Processing {data_source} data...")

        for _, game in game_iter:
            if for_training:
                season = game['Season']
                team1 = game['WTeamID']
                team2 = game['LTeamID']
                result = 1  # 1 means team1 won
            else:
                # Parse ID (e.g., "2024_1101_1102")
                id_parts = game['ID'].split('_')
                if len(id_parts) != 3:
                    continue
                season = int(id_parts[0])
                team1 = int(id_parts[1])
                team2 = int(id_parts[2])
                result = None  # No result for test data

            # Get team features for this season
            team1_features = team_features[(team_features['Season'] == season) &
                                          (team_features['TeamID'] == team1)]
            team2_features = team_features[(team_features['Season'] == season) &
                                          (team_features['TeamID'] == team2)]

            if len(team1_features) == 0 or len(team2_features) == 0:
                if not for_training:
                    # For test data, try to use most recent season's data
                    available_seasons = team_features[team_features['TeamID'] == team1]['Season'].tolist()
                    if available_seasons:
                        most_recent = max(available_seasons)
                        team1_features = team_features[(team_features['Season'] == most_recent) &
                                                      (team_features['TeamID'] == team1)]

                    available_seasons = team_features[team_features['TeamID'] == team2]['Season'].tolist()
                    if available_seasons:
                        most_recent = max(available_seasons)
                        team2_features = team_features[(team_features['Season'] == most_recent) &
                                                      (team_features['TeamID'] == team2)]

                if len(team1_features) == 0 or len(team2_features) == 0:
                    continue

            team1_features = team1_features.iloc[0]
            team2_features = team2_features.iloc[0]

            # Head-to-head results from regular season
            h2h_games = reg_season[(reg_season['Season'] == season) &
                                   (((reg_season['WTeamID'] == team1) & (reg_season['LTeamID'] == team2)) |
                                    ((reg_season['WTeamID'] == team2) & (reg_season['LTeamID'] == team1)))]

            team1_h2h_wins = len(h2h_games[h2h_games['WTeamID'] == team1])
            team2_h2h_wins = len(h2h_games[h2h_games['WTeamID'] == team2])

            # Create feature vector
            features = {
                'Season': season,
                'Team1ID': team1,
                'Team2ID': team2,
                'Team1Name': team1_features['TeamName'],
                'Team2Name': team2_features['TeamName'],

                # Team 1 features
                'Team1_WinPct': team1_features['WinPct'],
                'Team1_AvgPointsFor': team1_features['AvgPointsFor'],
                'Team1_AvgPointsAgainst': team1_features['AvgPointsAgainst'],
                'Team1_PointDiff': team1_features['PointDiff'],
                'Team1_NumGames': team1_features['NumGames'],
                'Team1_Seed': team1_features['Seed'],

                # Team 2 features
                'Team2_WinPct': team2_features['WinPct'],
                'Team2_AvgPointsFor': team2_features['AvgPointsFor'],
                'Team2_AvgPointsAgainst': team2_features['AvgPointsAgainst'],
                'Team2_PointDiff': team2_features['PointDiff'],
                'Team2_NumGames': team2_features['NumGames'],
                'Team2_Seed': team2_features['Seed'],

                # Head-to-head
                'H2H_Team1Wins': team1_h2h_wins,
                'H2H_Team2Wins': team2_h2h_wins,

                # Difference features
                'Diff_WinPct': team1_features['WinPct'] - team2_features['WinPct'],
                'Diff_PointDiff': team1_features['PointDiff'] - team2_features['PointDiff'],
                'Diff_Seed': (team2_features['Seed'] if not pd.isna(team2_features['Seed']) else 16) -
                            (team1_features['Seed'] if not pd.isna(team1_features['Seed']) else 16),

                # Same conference?
                'SameConference': 1 if team1_features['Conference'] == team2_features['Conference'] else 0
            }

            # Add ID for test data
            if not for_training:
                features['ID'] = game['ID']

            # Add result for training data
            if for_training:
                features['Result'] = result

            # Add Massey ratings if available
            if gender == 'M' and self.massey is not None:
                for system in ['POM', 'SAG', 'MOR', 'WLK']:
                    massey_col = f'Massey_{system}'
                    if massey_col in team1_features.index:
                        features[f'Team1_{massey_col}'] = team1_features[massey_col]
                        features[f'Team2_{massey_col}'] = team2_features[massey_col]
                        if not pd.isna(team1_features[massey_col]) and not pd.isna(team2_features[massey_col]):
                            features[f'Diff_{massey_col}'] = team2_features[massey_col] - team1_features[massey_col]
                        else:
                            features[f'Diff_{massey_col}'] = np.nan

            games.append(features)

        return pd.DataFrame(games)

    def prepare_training_data(self, gender='M'):
        """Prepare the complete training dataset"""
        print(f"\nPreparing {gender} training data...")

        # Create game features for training
        games_df = self.create_game_features(gender, for_training=True)

        if len(games_df) == 0:
            print(f"No training games found for {gender}")
            return pd.DataFrame()

        # Add the opposite matchup to double the data
        games_swapped = games_df.copy()
        games_swapped['Team1ID'] = games_df['Team2ID']
        games_swapped['Team2ID'] = games_df['Team1ID']
        games_swapped['Team1Name'] = games_df['Team2Name']
        games_swapped['Team2Name'] = games_df['Team1Name']
        games_swapped['Result'] = 0  # Now team1 (original team2) lost

        # Swap all team-specific features
        for col in games_df.columns:
            if col.startswith('Team1_') and col not in ['Team1ID', 'Team1Name']:
                feature = col[6:]  # Remove 'Team1_'
                team2_col = f'Team2_{feature}'
                if team2_col in games_df.columns:
                    games_swapped[col] = games_df[team2_col].values
                    games_swapped[team2_col] = games_df[col].values

        # Swap difference features
        for col in games_df.columns:
            if col.startswith('Diff_'):
                if col in games_swapped.columns:
                    games_swapped[col] = -games_df[col].values

        # Combine original and swapped
        full_dataset = pd.concat([games_df, games_swapped], ignore_index=True)

        # Remove rows with missing seeds (non-tournament teams in training data)
        full_dataset = full_dataset.dropna(subset=['Team1_Seed', 'Team2_Seed'])

        return full_dataset

    def prepare_test_data(self, gender='M'):
        """Prepare the complete test dataset"""
        print(f"\nPreparing {gender} test data...")

        # Create game features for testing
        test_df = self.create_game_features(gender, for_training=False)

        if len(test_df) == 0:
            print(f"No test games found for {gender}")
            return pd.DataFrame()

        return test_df

    def generate_all_data(self):
        """Generate training and testing data for both men and women"""
        print("="*60)
        print("MARCH MACHINE LEARNING MANIA - TRAINING & TESTING DATA GENERATION")
        print("="*60)

        # Load all data
        self.load_all_data()

        # Generate men's data
        print("\n" + "="*60)
        print("MEN'S TOURNAMENT DATA")
        print("="*60)

        # Men's training data
        print("\n" + "-"*40)
        men_training = self.prepare_training_data('M')
        if len(men_training) > 0:
            men_training.to_csv('men_training_data.csv', index=False)
            print(f"Men's training data saved to men_training_data.csv")
            print(f"Shape: {men_training.shape}")
            print(f"Training games: {len(men_training)}")
            print(f"Features: {len(men_training.columns)}")

        # Men's test data
        print("\n" + "-"*40)
        men_test = self.prepare_test_data('M')
        if len(men_test) > 0:
            men_test.to_csv('men_test_data.csv', index=False)
            print(f"Men's test data saved to men_test_data.csv")
            print(f"Shape: {men_test.shape}")
            print(f"Test matchups: {len(men_test)}")
            print(f"Features: {len(men_test.columns)}")

        # Generate women's data
        print("\n" + "="*60)
        print("WOMEN'S TOURNAMENT DATA")
        print("="*60)

        try:
            # Women's training data
            print("\n" + "-"*40)
            women_training = self.prepare_training_data('W')
            if len(women_training) > 0:
                women_training.to_csv('women_training_data.csv', index=False)
                print(f"Women's training data saved to women_training_data.csv")
                print(f"Shape: {women_training.shape}")
                print(f"Training games: {len(women_training)}")
                print(f"Features: {len(women_training.columns)}")

            # Women's test data
            print("\n" + "-"*40)
            women_test = self.prepare_test_data('W')
            if len(women_test) > 0:
                women_test.to_csv('women_test_data.csv', index=False)
                print(f"Women's test data saved to women_test_data.csv")
                print(f"Shape: {women_test.shape}")
                print(f"Test matchups: {len(women_test)}")
                print(f"Features: {len(women_test.columns)}")

        except Exception as e:
            print(f"Could not generate women's data: {e}")

        print("\n" + "="*60)
        print("DATA GENERATION COMPLETE!")
        print("="*60)

        return men_training, men_test



In [19]:
  generator = MarchMadnessDataGenerator(data_path="/content/drive/MyDrive/march-machine-learning-mania-2026/")
  training_data, test_data = generator.generate_all_data()

  if len(training_data) > 0 and len(test_data) > 0:
  # Display samples of the generated data
    print("\n" + "="*60)
    print("SAMPLE DATA PREVIEW")
    print("="*60)

    print("\nTraining Data Sample (first 5 rows):")
    train_cols = ['Season', 'Team1Name', 'Team2Name', 'Result',
                     'Team1_Seed', 'Team2_Seed', 'Diff_PointDiff']
    available_train_cols = [col for col in train_cols if col in training_data.columns]
    print(training_data[available_train_cols].head())

    print("\nTest Data Sample (first 5 rows):")
    test_cols = ['ID', 'Season', 'Team1Name', 'Team2Name',
                    'Team1_Seed', 'Team2_Seed', 'Diff_PointDiff']
    available_test_cols = [col for col in test_cols if col in test_data.columns]
    print(test_data[available_test_cols].head())

MARCH MACHINE LEARNING MANIA - TRAINING & TESTING DATA GENERATION
Loading all data files...
SampleSubmissionStage1.csv loaded
SampleSubmissionStage2.csv loaded
Massey Ordinals loaded with columns: ['Season', 'RankingDayNum', 'SystemName', 'TeamID', 'OrdinalRank']
All files loaded successfully!

MEN'S TOURNAMENT DATA

----------------------------------------

Preparing M training data...
Creating game features for M training...
Creating team features for M...
Adding Massey ratings...
Processing tournament data...
Men's training data saved to men_training_data.csv
Shape: (2630, 36)
Training games: 2630
Features: 36

----------------------------------------

Preparing M test data...
Creating game features for M testing...
Creating team features for M...
Adding Massey ratings...
Processing sample submission data...
Men's test data saved to men_test_data.csv
Shape: (64980, 36)
Test matchups: 64980
Features: 36

WOMEN'S TOURNAMENT DATA

----------------------------------------

Preparing W t